# Lab 3 · Fine-Tuning a Pretrained Transformer
**MSACL · DS301 Deep Learning · Segment 3 · Lab 3**

In Lecture 8 you saw the big idea of **transfer learning**: instead of training a giant
network from scratch, you download one that a lab with thousands of GPUs already trained on
millions of examples, snap a **fresh head** on top, and fine-tune it on *your* small dataset.

This lab makes that payoff visible. You'll take **DistilBERT** — a language model that read
~3 billion words of English — and fine-tune it to sort **medical abstracts** into 5 disease
categories. Then you'll run the **same recipe three ways** and watch the difference:

| Run | What we train | Expected accuracy |
|---|---|---|
| **Full fine-tune** | the whole pretrained model | **~65%** |
| **Frozen body, head only** | just the new classifier head | in between |
| **From scratch** | same architecture, *no* pretraining | **~33%** (majority floor) |

The gap between the first and the last row **is** the value of pretraining.

> **You fill in three short blanks** — attaching the head, choosing what to freeze, and the
> fine-tuning learning rate. Everything else runs for you. Look for **YOUR TURN ✏️**.

**Runtime:** on a Colab **T4 GPU** the three runs finish in ~6–9 min. Set the runtime with
**Runtime → Change runtime type → T4 GPU** before you start.


## Step 0 · Set up *(runs for you)*
On Colab the deep-learning libraries need one `pip install`. This is a no-op if they're
already present.


In [ ]:
# Read and run — no need to edit. (~30 s on a fresh Colab runtime)
# On Colab these install; locally they're usually already present.
try:
    import transformers, datasets, sklearn  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "datasets", "scikit-learn", "accelerate"], check=True)
print("Libraries ready.")

In [ ]:
# Read and run — no need to edit.
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoConfig,
                          AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score
import transformers

transformers.logging.set_verbosity_error()   # quiet the long load reports
np.random.seed(0)
torch.manual_seed(0)

DEVICE = "GPU (fp16)" if torch.cuda.is_available() else "CPU/MPS (fp32)"
print("PyTorch:", torch.__version__, "| running on:", DEVICE)

## Step 1 · Run-size settings *(runs for you)*
These few numbers cap the runtime so everything finishes inside a lab session. The full course
settings below are tuned for a **T4 GPU**. (If you're on a slow CPU, shrink `N_TRAIN`/`EPOCHS`.)
`FP16` — half-precision, 4–8× faster — turns on automatically only when a GPU is present.


In [ ]:
# Read and run — no need to edit. (Colab / T4 settings)
N_TRAIN    = 4000     # training abstracts (subsampled from 11,550 to keep it fast)
N_EVAL     = 1000     # held-out abstracts for honest evaluation
EPOCHS     = 3        # passes over the training set
MAX_LENGTH = 256      # tokens kept per abstract (longer = slower)
BATCH_SIZE = 32
FINETUNE_LR = None    # <- you set this in Step 5
SCRATCH_LR  = 5e-4    # the from-scratch run needs a MUCH bigger step (nothing is pretrained yet)
FP16 = torch.cuda.is_available()   # half precision only makes sense on a GPU
ATTN = "eager"   # attention kernel that runs on Colab GPU, Apple MPS, and CPU alike
                 # (MPS's fast default kernel cannot apply attention dropout while training)

# ESM-2 protein bonus (Step 8)
N_PEPTIDES = 3000
ESM_EPOCHS = 3

CHECKPOINT = "distilbert-base-uncased"   # the pretrained English model (Apache-2.0, no login)
NUM_LABELS = 5                           # 5 disease categories
print(f"train={N_TRAIN}  eval={N_EVAL}  epochs={EPOCHS}  fp16={FP16}")

## Step 2 · The data: `medical_abstracts` *(runs for you)*

**`TimSchopf/medical_abstracts`** is a public Hugging Face dataset of PubMed-style abstracts,
each labelled with one of **5 disease categories** — the kind of free-text triage a clinical
informatics team actually faces.

- **Source / license:** `TimSchopf/medical_abstracts` on Hugging Face, **CC-BY-SA-3.0**. No login.
- **Size:** 11,550 train / 2,888 test abstracts.
- **⚠️ Label gotcha (verified):** the raw `condition_label` is **1–5**, but PyTorch wants class
  ids **0–4**. We **subtract 1**. (Forget this and CUDA throws an opaque index error.)
- The human-readable class names live in a *separate* config, `"labels"`.


In [ ]:
# Read and run — no need to edit.
raw = load_dataset("TimSchopf/medical_abstracts", "default")

# class-id -> disease name (from the separate "labels" config); ids there are also 1..5
name_rows = load_dataset("TimSchopf/medical_abstracts", "labels")["train"]
CLASS_NAMES = [None] * 5
for r in name_rows:
    CLASS_NAMES[r["condition_label"] - 1] = r["condition_name"]

# remap the 1..5 label to a 0..4 class id in a new "labels" column (what the model expects)
def remap_labels(batch):
    return {"labels": [c - 1 for c in batch["condition_label"]]}

raw = raw.map(remap_labels, batched=True)

print("splits:", {k: len(v) for k, v in raw.items()})
print("classes (id -> name):")
for i, nm in enumerate(CLASS_NAMES):
    print(f"  {i}: {nm}")
print("\nexample abstract (first 220 chars):")
print(" ", raw["train"][0]["medical_abstract"][:220], "...")

Let's confirm the remap worked and look at the class balance — the biggest class sets the
**majority-vote floor** every model must beat to prove it learned anything.


In [ ]:
# Read and run — no need to edit.
train_label_ids = np.array(raw["train"]["labels"])
counts = np.bincount(train_label_ids, minlength=5)
majority_floor = counts.max() / counts.sum()

plt.figure(figsize=(7, 3))
plt.bar([CLASS_NAMES[i] for i in range(5)], counts)
plt.axhline(counts.max(), ls="--", color="gray")
plt.ylabel("# training abstracts"); plt.xticks(rotation=20, ha="right")
plt.title(f"Class balance — majority-vote floor = {majority_floor:.0%}")
plt.tight_layout(); plt.show()

# --- self-checks: label remap is correct (holds on any run size) ---
assert set(np.unique(train_label_ids)).issubset({0, 1, 2, 3, 4}), \
    "labels must be class ids 0..4 (did the 'subtract 1' remap run?)"
assert len(CLASS_NAMES) == 5 and all(CLASS_NAMES), "should have 5 named classes"
print(f"Labels are 0..4 across {len(train_label_ids)} abstracts. Majority floor = {majority_floor:.1%}.")

## Step 3 · Tokenize *(runs for you)*
A transformer reads **token ids**, not raw text (Lecture 7). We load DistilBERT's own tokenizer,
subsample to `N_TRAIN`/`N_EVAL` so the lab stays fast, and turn each abstract into ids
(truncated to `MAX_LENGTH`).


In [ ]:
# Read and run — no need to edit.
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["medical_abstract"], truncation=True, max_length=MAX_LENGTH)

train_ds = raw["train"].shuffle(seed=0).select(range(N_TRAIN)).map(tokenize, batched=True)
eval_ds  = raw["test"].shuffle(seed=0).select(range(N_EVAL)).map(tokenize, batched=True)

# a collator pads each batch to its own longest sequence (faster than padding everything to 256)
collator = DataCollatorWithPadding(tokenizer)

print(f"tokenized: {len(train_ds)} train / {len(eval_ds)} eval abstracts")
print("first abstract ->", len(train_ds[0]["input_ids"]), "tokens; first 12 ids:",
      train_ds[0]["input_ids"][:12])

## Step 4 · Attach a head & choose what to freeze — YOUR TURN ✏️

One `build_model` function makes all three networks. Two of your three blanks live here.

**Blank 1 — attach a fresh 5-class head.** `AutoModelForSequenceClassification.from_pretrained`
downloads the pretrained DistilBERT **body** and bolts a **brand-new, randomly-initialized
classification head** on top. You only have to tell it how many classes with `num_labels=`.
Use the constants `CHECKPOINT` and `NUM_LABELS`. Store it in `model`.

**Blank 2 — freeze the body (for the "head only" run).** Transfer learning's cheapest mode:
leave the pretrained body's weights *fixed* and train **only** the new head. In PyTorch you
freeze a weight by setting `requires_grad = False`. Loop over **`model.distilbert.parameters()`**
(the body) and switch each one off.

> The from-scratch model is built for you a few lines down with `from_config` — same
> architecture, but weights start **random** instead of pretrained.


In [ ]:
def build_model(mode):
    """mode: 'full' (fine-tune all), 'frozen' (train head only), or 'scratch' (no pretraining)."""
    if mode == "scratch":
        # same architecture, but RANDOM weights (no pretraining) — built for you
        config = AutoConfig.from_pretrained(CHECKPOINT, num_labels=NUM_LABELS)
        config._attn_implementation = ATTN
        return AutoModelForSequenceClassification.from_config(config)

    # ---- Blank 1: download the pretrained body + attach a fresh NUM_LABELS-way head ----
    ### BEGIN SOLUTION
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, num_labels=NUM_LABELS, attn_implementation=ATTN)
    ### END SOLUTION

    if mode == "frozen":
        # ---- Blank 2: freeze every weight in the DistilBERT BODY (train only the head) ----
        ### BEGIN SOLUTION
        for param in model.distilbert.parameters():
            param.requires_grad = False
        ### END SOLUTION

    return model


def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

# build all three now so we can check them before the (slower) training
models = {m: build_model(m) for m in ["full", "frozen", "scratch"]}

# --- self-checks: STRUCTURE only (true even before any training) ---
# (a) the head produces 5 logits per abstract
dummy = {k: torch.tensor(v).unsqueeze(0)
         for k, v in tokenizer("a short abstract", truncation=True, max_length=16).items()}
with torch.no_grad():
    logits = models["full"](**dummy).logits
assert tuple(logits.shape) == (1, NUM_LABELS), f"head must emit [batch, 5]; got {tuple(logits.shape)}"

# (b) full fine-tune: everything trainable; frozen: ONLY the 594,437-param head trainable
full_tr,  full_tot  = count_trainable(models["full"])
froz_tr,  froz_tot  = count_trainable(models["frozen"])
assert full_tr == full_tot, "full fine-tune should train every parameter"
assert froz_tr == 594437, f"frozen run should train only the 594,437 head params; got {froz_tr:,}"
assert not any(p.requires_grad for p in models["frozen"].distilbert.parameters()), \
    "the DistilBERT body must be frozen in the 'frozen' run"

# (c) from-scratch really is un-pretrained: its body weights differ from the pretrained ones
emb_full    = models["full"].distilbert.embeddings.word_embeddings.weight
emb_frozen  = models["frozen"].distilbert.embeddings.word_embeddings.weight
emb_scratch = models["scratch"].distilbert.embeddings.word_embeddings.weight
assert torch.equal(emb_full, emb_frozen), "full & frozen should share the SAME pretrained weights"
assert not torch.equal(emb_full, emb_scratch), "from-scratch weights must be RANDOM, not pretrained"

print(f"full fine-tune : {full_tr:>12,} / {full_tot:,} trainable")
print(f"frozen head    : {froz_tr:>12,} / {froz_tot:,} trainable  (just the head!)")
print(f"from scratch   : {count_trainable(models['scratch'])[0]:>12,} trainable  (random init)")

## Step 5 · The fine-tuning learning rate — YOUR TURN ✏️

**Blank 3.** When you fine-tune, the body's weights are *already good* — you only want to nudge
them. So the fine-tuning learning rate is **much smaller** than you'd use from scratch. A blunt
step would smash the very knowledge you're trying to reuse.

Set `FINETUNE_LR` to a typical fine-tuning value: **`2e-5`** (i.e. 0.00002). Compare it to
`SCRATCH_LR = 5e-4` (Step 1) — the from-scratch run needs a step **~25× bigger** because it starts
from nothing.

**💬 Why smaller?** (jot your answer): *the pretrained weights already encode language; small
steps refine them without erasing what the model learned from 3 billion words.*


In [ ]:
# ---- Blank 3: a small fine-tuning learning rate (much smaller than from-scratch) ----
### BEGIN SOLUTION
FINETUNE_LR = 2e-5
### END SOLUTION

# --- self-check: fine-tuning lr is far below a from-scratch lr (magnitude, not exact value) ---
assert FINETUNE_LR is not None, "set FINETUNE_LR"
assert FINETUNE_LR < 1e-3, "fine-tuning lr should be MUCH smaller than a from-scratch lr (<1e-3)"
assert FINETUNE_LR < SCRATCH_LR, "fine-tuning should step more gently than the from-scratch run"
print(f"fine-tuning lr = {FINETUNE_LR:g}   vs   from-scratch lr = {SCRATCH_LR:g}"
      f"   ({SCRATCH_LR / FINETUNE_LR:.0f}x bigger)")

## Step 6 · A reusable training helper *(runs for you)*

We use Hugging Face's **`Trainer`**, which wraps the exact same forward → loss → backward →
step loop from Lab 1 (it just handles batching, the GPU, and fp16 for us). This one helper runs
any of the three models so the comparison is truly apples-to-apples.


In [ ]:
# Read and run — no need to edit.
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(pred.label_ids, preds)}

def run_experiment(mode, lr):
    model = models[mode]
    args = TrainingArguments(
        output_dir=f"/tmp/lab03_{mode}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=lr,
        fp16=FP16,
        eval_strategy="epoch",
        logging_strategy="no",
        save_strategy="no",
        report_to="none",
        seed=0,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=train_ds, eval_dataset=eval_ds,
                      data_collator=collator, compute_metrics=compute_metrics)
    trainer.train()
    return trainer.evaluate()["eval_accuracy"]

print("Training helper ready.")

## Step 7 · The three-way comparison *(runs for you)*

Now the payoff. We train all three and record test accuracy. On a T4 this is the ~6–9 min part
— watch the progress bars.

- **full** and **frozen** use your small `FINETUNE_LR`;
- **scratch** uses the bigger `SCRATCH_LR` (it has to learn language *and* the task from zero).


In [ ]:
# Read and run — no need to edit.
results = {}
results["full fine-tune"]      = run_experiment("full",    FINETUNE_LR)
results["frozen (head only)"]  = run_experiment("frozen",  FINETUNE_LR)
results["from scratch"]        = run_experiment("scratch", SCRATCH_LR)

# --- self-check: STRUCTURE of the comparison (not the accuracy magnitudes) ---
assert len(results) == 3, "the comparison should hold exactly three runs"
assert all(0.0 <= v <= 1.0 for v in results.values()), "every accuracy is a fraction 0..1"
print("\nTest accuracy:")
for name, acc in results.items():
    print(f"  {name:<20} {acc:.3f}")

### Read the bar plot
Full fine-tuning should tower over from-scratch; the frozen head-only run lands in between — you
got most of the benefit by training **&lt;1%** of the parameters. From-scratch hugs the
**majority-vote floor**: with only 4,000 abstracts and no pretraining, DistilBERT can't learn
English *and* the task at once. **That gap is the entire lesson of transfer learning.**


In [ ]:
# Read and run — no need to edit.
names = list(results)
accs  = [results[n] for n in names]
colors = ["#2a7", "#59c", "#c66"]

plt.figure(figsize=(7, 4))
bars = plt.bar(names, accs, color=colors)
plt.axhline(majority_floor, ls="--", color="gray")
plt.text(2.4, majority_floor + 0.01, f"majority floor {majority_floor:.0%}",
         ha="right", color="gray", fontsize=9)
plt.ylim(0, 1); plt.ylabel("test accuracy")
plt.title("Pretraining is the game-changer")
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.0%}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

## What just happened
You ran the transfer-learning recipe from Lecture 8 for real:

- **attached a fresh head** onto a pretrained body (`num_labels=5`),
- **froze the body** to train only the head — most of the benefit for &lt;1% of the parameters,
- used a **small fine-tuning learning rate** so you refine, not erase, the pretrained weights,
- and saw **from-scratch collapse to the majority floor** — proof that the model's power comes
  from what it learned *before* it ever saw your data.

The headline: with a pretrained model, **a few thousand labelled examples is often enough**.


## Step 8 · Bonus — the *same move, a different alphabet* 🧬

DistilBERT read ~3 billion words of **English**. **ESM-2** read ~250 million **protein
sequences** — a language model whose "words" are amino acids. The fine-tuning recipe is
*identical*; only the alphabet changes.

We'll fine-tune the tiny **ESM-2 (8M)** to flag **cationic peptides** — a crude stand-in for the
antimicrobial-peptide question, since real AMPs tend to carry a high positive net charge.

> **This peptide set is SYNTHETIC**, generated deterministically in this notebook (no download,
> no license worries). Rule: a peptide is **class 1 ("cationic")** if its net charge
> `(#K + #R) − (#D + #E) ≥ 3`, else **class 0**. It's a demo of the *recipe*, not a graded task.


In [ ]:
# Read and run — no need to edit.  Deterministic synthetic peptide set.
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")   # the 20 standard amino acids
rng = np.random.default_rng(0)               # fixed seed -> identical peptides every run

def net_charge(pep):
    return sum(pep.count(a) for a in "KR") - sum(pep.count(a) for a in "DE")

# draw random peptides until we have a class-BALANCED set of N_PEPTIDES
# (cationic peptides are the minority under random sampling, so we collect each class
#  separately until both halves are full)
half = N_PEPTIDES // 2
pos, neg = [], []
while len(pos) < half or len(neg) < half:
    length = int(rng.integers(12, 26))
    pep = "".join(rng.choice(AMINO_ACIDS, size=length))
    if net_charge(pep) >= 3:
        if len(pos) < half:
            pos.append(pep)
    elif len(neg) < half:
        neg.append(pep)

peptides = pos + neg
pep_labels = [1] * half + [0] * half
order = rng.permutation(len(peptides))
peptides   = [peptides[i] for i in order]
pep_labels = [pep_labels[i] for i in order]

n1 = int(np.sum(pep_labels))
print(f"{len(peptides)} synthetic peptides  ({n1} cationic / {len(peptides) - n1} not)")
print("example:", peptides[0], "-> net charge", net_charge(peptides[0]),
      "-> class", pep_labels[0])

# --- self-checks: STRUCTURE of the synthetic task ---
assert set(pep_labels) == {0, 1}, "peptide task should be binary (0/1)"
assert all(net_charge(p) >= 3 for p, y in zip(peptides, pep_labels) if y == 1), \
    "every class-1 peptide must satisfy the net-charge >= 3 rule"
assert all(net_charge(p) < 3 for p, y in zip(peptides, pep_labels) if y == 0), \
    "every class-0 peptide must break the net-charge >= 3 rule"
print("Synthetic peptide labels follow the charge rule exactly.")

In [ ]:
# Read and run — no need to edit.  Same recipe: tokenize -> attach head -> fine-tune.
from datasets import Dataset as HFDataset

ESM_CKPT = "facebook/esm2_t6_8M_UR50D"    # 8M-param protein language model (MIT license)
esm_tok = AutoTokenizer.from_pretrained(ESM_CKPT)

n_pep_train = int(0.8 * len(peptides))
pep_ds = HFDataset.from_dict({"seq": peptides, "labels": pep_labels})
pep_ds = pep_ds.map(lambda b: esm_tok(b["seq"], truncation=True, max_length=32), batched=True)
pep_train = pep_ds.select(range(n_pep_train))
pep_eval  = pep_ds.select(range(n_pep_train, len(peptides)))

esm_model = AutoModelForSequenceClassification.from_pretrained(ESM_CKPT, num_labels=2, attn_implementation=ATTN)  # fresh 2-way head

esm_args = TrainingArguments(
    output_dir="/tmp/lab03_esm", num_train_epochs=ESM_EPOCHS,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=FINETUNE_LR, fp16=FP16, eval_strategy="epoch",
    logging_strategy="no", save_strategy="no", report_to="none", seed=0)
esm_trainer = Trainer(model=esm_model, args=esm_args,
                      train_dataset=pep_train, eval_dataset=pep_eval,
                      data_collator=DataCollatorWithPadding(esm_tok),
                      compute_metrics=compute_metrics)
esm_trainer.train()
esm_acc = esm_trainer.evaluate()["eval_accuracy"]

pep_floor = max(np.mean(pep_labels), 1 - np.mean(pep_labels))   # majority-vote baseline
print(f"\nESM-2 fine-tuned accuracy: {esm_acc:.2f}   (majority baseline {pep_floor:.2f})")

# --- self-check: it ran and returned a valid accuracy (magnitude shown, not asserted) ---
assert 0.0 <= esm_acc <= 1.0, "accuracy must be a fraction"
print("Same recipe, different alphabet: DistilBERT read English; ESM-2 read proteins.")

## Discussion D3 (10 min)

**What pretrained model could *your* product borrow — and what would "the head" predict?**

Think about your own assay or instrument:
- Is there a **pretrained backbone** that already speaks your data's language? (a protein LM like
  ESM-2 for peptides; a spectra/CNN backbone from Lab 2; a chemistry LM for SMILES; a general
  vision model for MS-imaging.)
- What would **the new head** output — resistant/susceptible, a QC pass/fail, a disease class, a
  retention-time number?
- You have maybe a few hundred labelled examples. From today's plot, does **fine-tune vs. from
  scratch** even look like a contest? What would you freeze, and why?

Jot a one-line "pretrained backbone → my head" idea to share with a neighbour.
